# Project 3
Maja Domańska i Martyna Pawlak

#### Imports

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import squidpy as sq
from scipy.spatial import KDTree

adata = ad.read_h5ad("data/train_adata.h5ad")
adata.layers["exprs_arcsinh"] = adata.layers["exprs"]

## Task 1 - Dataset overview

In [ ]:
overview = pd.DataFrame({"value": [adata.n_obs, adata.n_vars, adata.obs["image"].nunique(), adata.obs["Indication"].nunique()]})
overview.index = ["Number of cells", "Number of markers", "Number of images", "Number of cancer indications"]

display(overview)

In [ ]:
celltype_counts = adata.obs["celltypes"].value_counts()
display(celltype_counts.to_frame("count"))

In [ ]:
tumor_fraction = (adata.obs["celltypes"] == "Tumor").mean() * 100
print(f"Tumor fraction: {tumor_fraction:.1f}%")


## Task 2 - Marker expression distributions 

In [ ]:
markers = ["Ecad", "CD8a", "Ki67"]

# three subplots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, marker in zip(axes, markers):
    col = np.where((adata.var["marker"] == marker) & (adata.var["use_channel"] == 1))[0][0]

    expr = adata.layers["exprs_arcsinh"][:, col]

    ax.hist(expr, bins=80)
    ax.set_title(marker)
    ax.set_xlabel("arcsinh-transformed expression")
    ax.set_ylabel("Number of cells")

plt.tight_layout()
plt.show()

Ecad: Distribution is bimodal, indicating two distinct expression states of Ecad across the dataset, this suggests substantial heterogeneity in Ecad expression among cells.

CD8a: Distribution is unimodal with a long right tail, suggesting that only a subset of cells exhibits elevated CD8a expression, this indicates cell-type specificity of the marker.

Ki67: Distribution is unimodal with a long right tail, indicating that most cells have low Ki67 expression while only a subset shows elevated expression, this suggests that proliferative activity is limited to a fraction of the cell population.

## Task 3 - Tumour burden, marker expression, and indication clustering

In [ ]:
tumor_fraction_ind = (
    adata.obs
    .assign(is_tumor=adata.obs["celltypes"] == "Tumor")
    .groupby("Indication", observed=True)["is_tumor"]
    .mean()
    .sort_values(ascending=False)
)

# barplot of tumor cell fraction per indication
plt.figure(figsize=(9, 5))
tumor_fraction_ind.plot(kind="bar")
plt.ylabel("Tumor cell fraction")
plt.xlabel("Indication")
plt.title("Tumor cell fraction per indication")
plt.tight_layout()
plt.show()

In [ ]:
tumor = adata[adata.obs["celltypes"] == "Tumor"].copy()

markers = ["Ecad", "CarbonicAnhydrase", "Ki67"]

# Table - rows are cancer indications and columns are marker names
marker_df = pd.DataFrame(index=tumor.obs["Indication"].unique(), columns=markers)

# For each marker, extract expression and calculate mean expression in tumor cells per indication
for marker in markers:
    col = np.where(tumor.var["marker"] == marker)[0][0]
    expr = tumor.layers["exprs_arcsinh"][:, col]
    marker_df[marker] = (pd.DataFrame({"Indication": tumor.obs["Indication"].values,"expr": expr}).groupby("Indication", observed=True)["expr"].mean())

marker_df = marker_df.astype(float)
display(marker_df)

In [ ]:
from scipy.stats import zscore

# z-score
zdf = pd.DataFrame(zscore(marker_df.values, axis=0), index=marker_df.index, columns=marker_df.columns)

# heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(zdf, annot=True, cmap="vlag", center=0)
plt.title("Z-scored marker expression in Tumor cells")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

# Hierarchical clustering of cancer indications based on their mean tumour-marker expression profile
Z = linkage(marker_df, method="ward")

# dendogram 
plt.figure(figsize=(8, 5))
dendrogram(Z, labels=marker_df.index.tolist())
plt.title("Hierarchical clustering of indications")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

HN and BREAS cluster together most closely, while THOR and GI form a second cluster, GU is the most distinct indication and separates from all other cancer types at the largest clustering distance. HN and BREAS share relatively high Ecad and Ki67 expression, whereas THOR and GI show intermediate Ecad and Ki67 levels together with lower CarbonicAnhydrase expression, suggesting similar tumour molecular profiles within each cluster.

## Task 4 - Spatially resolved tissue map with marker overlay

In [ ]:
# occurrences of each cell type for each image
cell_counts = adata.obs.groupby(['image', 'celltypes'], observed=True).size().unstack(fill_value=0)

# limit the analysis to images with >= 100 tumor cells
valid_images = cell_counts[cell_counts['Tumor'] >= 100]

criterion = valid_images['CD8'] / valid_images['Tumor']

best_image_id = criterion.idxmax()
best_criterion_value = criterion.max()

print(f"image id: {best_image_id}")
print(f"criterion value: {best_criterion_value:.4f}")

In [ ]:
# data for the chosen best image
adata_img = adata[adata.obs['image'] == best_image_id].copy()

# marker of our choice
marker_to_plot = 'PDL1'

# we use the helper function given in the manual to retrieve the marker expression
def get_expr(adata_obj, marker):
    col = int(np.where(adata_obj.var['marker'] == marker)[0][0])
    return adata_obj.layers['exprs_arcsinh'][:, col]

marker_expr = get_expr(adata_img, marker_to_plot)

# clip to the 5th-95th percentile to avoid outlier saturation.
vmin = np.percentile(marker_expr, 5)
vmax = np.percentile(marker_expr, 95)

# two plots next to each other
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# left panel - scatter plot coloured by cell type
sns.scatterplot(
    data=adata_img.obs,
    x='Pos_X', 
    y='Pos_Y', 
    hue='celltypes',
    palette='tab20',
    s=15, 
    edgecolor='none',
    ax=axes[0]
)
axes[0].set_title('Spatial cell-type organisation')
axes[0].set_xlabel('Pos_X')
axes[0].set_ylabel('Pos_Y')
axes[0].invert_yaxis()
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', markerscale=2)

# right panel - scatter plot of all cells coloured by the continuous expression of chosen marker - with the use of virdis
sc = axes[1].scatter(
    x=adata_img.obs['Pos_X'],
    y=adata_img.obs['Pos_Y'],
    c=marker_expr,
    cmap='viridis',
    vmin=vmin,
    vmax=vmax,
    s=15,
    edgecolor='none'
)
axes[1].set_title(f'Continuous expression of {marker_to_plot}')
axes[1].set_xlabel('Pos_X')
axes[1].set_ylabel('Pos_Y')
axes[1].invert_yaxis()
plt.colorbar(sc, ax=axes[1], label=f'{marker_to_plot} expression')

plt.tight_layout()
plt.show()

##### Does the marker's spatial pattern align with or deviate from the cell-type boundaries?

The spatial pattern of PDL1 expression deviates from individual cell-type boundaries, but is localized in a specific place, suggesting that PDL1 is locally induced across multiple cell types in response to the microenvironment rather than being a baseline feature of a single cell type

## Task 5 - Neighbourhood enrichment

In [ ]:
mask = adata.obs['image'] == best_image_id
adata_s = adata[mask].copy()

adata_s.obsm['spatial'] = np.column_stack( [adata_s.obs['Pos_X'].values, adata_s.obs['Pos_Y'].values])

In [ ]:
sq.gr.spatial_neighbors(adata_s, coord_type='generic', spatial_key='spatial', n_neighs=10)

sq.gr.nhood_enrichment(adata_s, cluster_key='celltypes')

In [ ]:
# generate heat map
sq.pl.nhood_enrichment(
    adata_s, 
    cluster_key='celltypes',
    figsize=(10, 8),
    title=f'Neighbourhood enrichment z-scores (image: {best_image_id})'
)
plt.show()

# automatic extraction of text results
zscore = adata_s.uns['celltypes_nhood_enrichment']['zscore']
cell_type_names = adata_s.obs['celltypes'].cat.categories.tolist()

# create a DataFrame for easier analysis of the 'Tumor' row
zscore_df = pd.DataFrame(zscore, index=cell_type_names, columns=cell_type_names)
tumor_row = zscore_df.loc['Tumor']

# Tumor self-enrichment
tumor_self_z = tumor_row['Tumor']

# top-2 cell types closest to Tumor
tumor_next_to = tumor_row.drop('Tumor').sort_values(ascending=False)
top_1_cell = tumor_next_to.index[0]
top_1_z = tumor_next_to.iloc[0]
top_2_cell = tumor_next_to.index[1]
top_2_z = tumor_next_to.iloc[1]

# CD8 cells
cd8_z = tumor_row['CD8']
cd8_status = "enriched " if cd8_z > 0 else "depleted"

print(f"top-2 cell types most enriched next to tumor:")
print(f" first: {top_1_cell}, z-score = {top_1_z:.2f}")
print(f" second: {top_2_cell}, z-score = {top_2_z:.2f}")
print(f"tumor self-enrichment z-score: {tumor_self_z:.2f}")
print(f"CD8 cells around tumor are: {cd8_status} z-score = {cd8_z:.2f}")

## Task 6 - Immune composition around Tumor cells

In [ ]:
all_image_fractions = []
weights = []

# all available cell type categories (we keep the same order)
all_celltypes = adata.obs['celltypes'].cat.categories

# we iterate over each image in the training dataset
for img_id, sub_df in adata.obs.groupby('image', observed=True):
    coords_all = sub_df[['Pos_X', 'Pos_Y']].values
    tumor_mask = sub_df['celltypes'] == 'Tumor'
    n_tumor = tumor_mask.sum()
    
    if n_tumor == 0:
        continue
        
    tumor_coords = coords_all[tumor_mask]
    
    # build a KDTree on the coordinates of all cells from this image
    tree = KDTree(coords_all)
    
    # 11 nearest neighbors
    distances, indices = tree.query(tumor_coords, k=11)
    
    # discard the first column index (itself) and flatten the neighbor matrix
    neighbor_indices = indices[:, 1:].flatten()
    
    # read the cell types of our nearest neighbors
    neighbor_celltypes = sub_df['celltypes'].iloc[neighbor_indices]
    
    # calculate the number of occurrences and convert them to fractions (filling in the missing types with zeros)
    counts = neighbor_celltypes.value_counts().reindex(all_celltypes, fill_value=0)
    fractions = counts / counts.sum()
    
    # save the results from this image
    all_image_fractions.append(fractions)
    weights.append(n_tumor)

In [ ]:

# convert the list of series into one common df (rows = images, columns = cell types)
df_fractions = pd.DataFrame(all_image_fractions)
weights = np.array(weights)

# calculate the weighted average: sum(fraction * weight) / sum(weight)
weighted_means = (df_fractions.multiply(weights, axis=0)).sum(axis=0) / weights.sum()

# remove the 'Tumor' category and sort descending
weighted_means_no_tumor = weighted_means.drop('Tumor').sort_values(ascending=False)


In [ ]:

# horizontal barplot of weighted-mean neighbour cell-type fractions
plt.figure(figsize=(10, 6))
sns.barplot(x=weighted_means_no_tumor.values, y=weighted_means_no_tumor.index, palette='viridis')
plt.xlabel('Weighted-mean fraction')
plt.ylabel('Cell type')
plt.title('Local immune microenvironment of Tumor cells')
plt.tight_layout()
plt.show()

# automatic leader indication
most_frequent_immune = weighted_means_no_tumor.index[0]
highest_fraction = weighted_means_no_tumor.iloc[0]

print(f"the most common non-Tumor neighbor: {most_frequent_immune}")
print(f"weighted average fraction value: {highest_fraction:.4f}")

Across all 132 images, CD8 is the most frequently adjacent non-Tumor cell type to Tumor cells, representing the predominant component of the immediate cellular neighborhood within the tumor microenvironment

## Task 7 - Tumour cell subclustering with model selection 

In [ ]:
# image selected in Task 4
image_id = "IMMUcan_Batch20210701_10075013-SPECT-VAR-TIS-UNST-03_002.tiff"

# keep only tumour cells from image
tumor_img = adata[(adata.obs["image"] == image_id) & (adata.obs["celltypes"] == "Tumor")].copy()

print(f"Number of tumor cells: {tumor_img.n_obs}")

In [ ]:
pd.DataFrame(adata.var[["marker", "use_channel"]])

##### Selected marker panel:  Ecad, CarbonicAnhydrase, Ki67, PDL1

Ecad, CarbonicAnhydrase, Ki67 and PDL1 were selected because they represent complementary aspects of tumour biology. Ecad reflects epithelial identity, CarbonicAnhydrase is associated with metabolic adaptation, Ki67 measures proliferative activity, and PDL1 is related to interactions with the immune microenvironment. These markers were also used in previous analyses (Tasks 3 and 4), making them suitable for identifying biologically distinct tumour subpopulations.

In [ ]:
markers = ["Ecad", "CarbonicAnhydrase", "Ki67", "PDL1"]

# extract marker expression
expr_matrix = np.column_stack([
 tumor_img.layers["exprs_arcsinh"][
        :,
        np.where(tumor_img.var["marker"] == marker)[0][0]
    ]
    for marker in markers
])

expr_matrix.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = StandardScaler().fit_transform(expr_matrix) # standardise marker expression

inertias = []
silhouettes = []

# evaluate k from 3 to 7
for k in range(3, 8):

    km = KMeans(n_clusters=k, random_state=42, n_init=10)

    km.fit(X)

    sil = silhouette_score(X, km.labels_, sample_size=min(len(X), 2000), random_state=42)

    inertias.append(km.inertia_)
    silhouettes.append(sil)

In [ ]:
# Elbow plot
plt.figure(figsize=(6, 4))

plt.plot(range(3, 8), inertias, marker="o")

plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow plot")

plt.tight_layout()
plt.show()

In [ ]:
#Silhouette plot
plt.figure(figsize=(6, 4))

plt.plot(range(3, 8), silhouettes, marker="o")

plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette analysis")

plt.tight_layout()
plt.show()

In [ ]:
print(pd.DataFrame({"k": range(3, 8), "inertia": inertias, "silhouette": silhouettes}))

##### Selected k = 4
Because it achieved the highest silhouette score and corresponds to the elbow point where further increases in k provide only modest reductions in inertia.

In [ ]:
# clustering
best_k = 4

km = KMeans(n_clusters=best_k, random_state=42, n_init=10)

tumor_img.obs["cluster"] = km.fit_predict(X)

In [ ]:
# spacial scatter plot
plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=tumor_img.obs,
    x="Pos_X",
    y="Pos_Y",
    hue="cluster",
    palette="tab10",
    s=20,
    edgecolor=None
)

plt.gca().invert_yaxis()

plt.title(f"Tumour cell subclusters (k = {best_k})")
plt.xlabel("Pos_X")
plt.ylabel("Pos_Y")

plt.tight_layout()
plt.show()

In [ ]:
# mean marker expression in clusters
cluster_profiles = pd.DataFrame(expr_matrix, columns=markers)
cluster_profiles["cluster"] = tumor_img.obs["cluster"].values
cluster_profiles.groupby("cluster").mean()

## Task 8 - Biological interpretation of Tumor subclusters 

In [ ]:
# Select usable biological markers only
usable_markers = tumor_img.var[
    tumor_img.var["use_channel"] == 1
]["marker"].tolist()

# Remove technical channels
usable_markers = [
    marker for marker in usable_markers
    if marker not in ["HistoneH3", "DNA1", "DNA2"]
]

print(len(usable_markers))
print(usable_markers)

In [ ]:
# Extract arcsinh-transformed expression for all markers
expr_all = pd.DataFrame({
    marker: tumor_img.layers["exprs_arcsinh"][
        :,
        np.where(tumor_img.var["marker"] == marker)[0][0]
    ]
    for marker in usable_markers
})

# Add cluster labels from Task 7
expr_all["cluster"] = tumor_img.obs["cluster"].values

expr_all.head()

In [ ]:
# Compute mean marker expression per Tumor subcluster
mean_expr = expr_all.groupby("cluster").mean()

mean_expr

In [ ]:
# Z-score each marker across clusters
mean_expr_z = mean_expr.T

mean_expr_z = mean_expr_z.sub(mean_expr_z.mean(axis=1), axis=0)
mean_expr_z = mean_expr_z.div(mean_expr_z.std(axis=1), axis=0)

# heatmap
plt.figure(figsize=(9, 11))
sns.heatmap(mean_expr_z, cmap="vlag", center=0)

plt.title("Mean marker expression per Tumor subcluster")
plt.xlabel("Cluster")
plt.ylabel("Marker")

plt.tight_layout()
plt.show()

##### Proposed labels:
##### Cluster 0 - Epithelial-like tumor cells
##### Cluster 1 - Proliferating tumor cells
##### Cluster 2 - Low-activity tumor cells
##### Cluster 3 - Immune-evasive and stressed tumor cells

Cluster 0 was characterised by high Ecad expression together with elevated TCF7 and CarbonicAnhydrase levels and was therefore labelled as epithelial-like.

Cluster 1 was labelled as proliferating because it showed the strongest Ki67 expression. 

Cluster 2 showed generally low expression of most analysed markers and was interpreted as a low-activity tumour-cell state.

Cluster 3 showed broad upregulation of immune-related markers, including several CD markers, together with high PDL1, Ido1 and VISTA, indicating an immune-marker-high and immune-evasive phenotype; increased cleavedPARP and SMA additionally suggest stress and a more invasive-like state.

These results indicate substantial molecular heterogeneity within the tumour, with distinct epithelial-like, proliferative, immune-evasive and low-activity tumour-cell states.

## Task 9 - Cross-cohort validation

### Chosen pattern:
The ranking and composition of non-Tumor immune and stromal cells in the immediate spatial neighborhood of Tumor cells (derived from Task 6).
### Hypothesis:
The pattern seen in the first dataset—where CD8 T cells directly surround and interact with tumor cells—will also be true in the new test dataset.

In [ ]:
# test set preparation
adata_test = ad.read_h5ad('data/test_adata.h5ad')
adata_test.layers['exprs_arcsinh'] = adata_test.layers['exprs']
adata_test = adata_test[adata_test.obs['celltypes'] != 'undefined'].copy()

# code from task 6 as a function to compare results
def get_tumor_neighborhood(adata_obj):
    all_image_fractions = []
    weights = []
    valid_celltypes = [ct for ct in adata_obj.obs['celltypes'].unique() if ct != 'undefined']
    
    for img_id, sub_df in adata_obj.obs.groupby('image', observed=True):
        coords = sub_df[['Pos_X', 'Pos_Y']].values
        tumor_mask = sub_df['celltypes'] == 'Tumor'
        n_tumor = tumor_mask.sum()
        
        if n_tumor == 0:
            continue
            
        tree = KDTree(coords)
        distances, indices = tree.query(coords[tumor_mask], k=11)
        neighbor_indices = indices[:, 1:].flatten()
        neighbor_celltypes = sub_df['celltypes'].iloc[neighbor_indices]
        
        counts = neighbor_celltypes.value_counts().reindex(valid_celltypes, fill_value=0)
        fractions = counts / counts.sum()
        all_image_fractions.append(fractions)
        weights.append(n_tumor)
        
    df_fractions = pd.DataFrame(all_image_fractions)
    weights = np.array(weights)
    weighted_means = (df_fractions.multiply(weights, axis=0)).sum(axis=0) / weights.sum()
    
    return weighted_means.drop('Tumor', errors='ignore').sort_values(ascending=False)

train_res = get_tumor_neighborhood(adata)
test_res = get_tumor_neighborhood(adata_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

# left plot - train
sns.barplot(x=train_res.values, y=train_res.index, palette='viridis', ax=axes[0])
axes[0].set_title('Train set (132 images) - local neighborhood of tumor cells')
axes[0].set_xlabel('Weighted-mean fraction')
axes[0].set_ylabel('Cell type')

# right plot - test
test_res_aligned = test_res.reindex(train_res.index).fillna(0)
sns.barplot(x=test_res_aligned.values, y=test_res_aligned.index, palette='viridis', ax=axes[1])
axes[1].set_title('Test set (47 images) - local neighborhood of tumor cells')
axes[1].set_xlabel('Weighted-mean fraction')
axes[1].set_ylabel('Cell type')

plt.tight_layout()
plt.show()

### Does the pattern replicate, where do the cohorts agree or disagree, and what is the biological implication?

The arrangement of cells around the tumor was not fully replicated between the two patient groups. In the training data, CD8 immune cells were the tumor's closest neighbor, while in the test data, it was the blood vessel cells (Mural). However, both cohorts agree that these two cell types play the most important role in the local tumor environment. Biologically, this means that in the training group, we encountered tumors in which lymphocytes directly attacked cancer cells. In the test group, the vascular component dominated, where the vascular wall cells could form a barrier separating the tumor from the immune system. These differences likely result from a different distribution of tumor types or different disease stages in the patients in the two cohorts.